In [2]:

import os
import sys

sys.path.append("..")

from ast import literal_eval

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from tqdm import tqdm
from shapely import wkt

from pipelines.utils import PLAIN_DATASET_NAME, PRE_MAP_DATASET_NAME, ROOT_DIR, DATASET_NAME, PEPROCESSED_DATASET_NAME, load_road_network
from preprocessing.utils import PREPROCESS_MAP

from preprocessing.rs_mapping import create_road_mapping_df, post_processing_mapped_df, merge_preprocessed_and_fmm

from preprocessing.cell_mapping import clean_and_output_data
from preprocessing.visualize import plot_gps_traj, plot_cpath
from pipelines.utils import load_config, generate_train_test_split, generate_train_val_test_split, load_road_network


config = load_config(name="cd", ctype="dataset")

In [ ]:
config

In [ ]:
# load original dataset
df = pd.read_csv(
    os.path.join("/data/shared/schestakov/data/china", "CD.csv")
)

In [ ]:
import pandas as pd
from shapely.geometry import LineString, Point
import time
import re # Using regex might be slightly faster for parsing
import os # To get CPU count
import math
from coord_convert.transform import gcj2wgs

# Import and initialize Pandarallel
from pandarallel import pandarallel


CREATE_LINESTRING = True
TOTAL_CORES = os.cpu_count()
NUM_WORKERS = 80 

print(f"System has {TOTAL_CORES} cores. Using {NUM_WORKERS} workers for parallel processing.")

# Initialize Pandarallel
pandarallel.initialize(nb_workers=NUM_WORKERS, progress_bar=True)


############## FOR CHENGDU WE NEED TO shift Points #### 
def shift_coords(coords, north_m=280, west_m=240):
    shifted = []
    for lon, lat in coords:
        # Shift latitude north
        dlat = north_m / 111320
        # Shift longitude west (negative direction)
        dlon = -west_m / (111320 * math.cos(math.radians(lat)))
        shifted.append((lon + dlon, lat + dlat))
    return shifted


def parse_trajectory_regex(traj_str, apply_shift=True):
    """
    Parses the trajectory string using regular expressions. Potentially faster.
    Returns timestamps, coordinates, and optionally a LineString.
    (Identical to the previous version)
    """
    if not isinstance(traj_str, str) or len(traj_str) < 5:
        return [], [], None

    timestamps = []
    coords = []
    # Pre-compile regex for slight performance boost if function is called many times
    # Although for parallel apply, the overhead might negate this benefit slightly.
    # Keep it simple here.
    pattern = re.compile(r'(\d+\.?\d*)\s+(\d+\.?\d*)\s+(\d+)')

    try:
        matches = pattern.findall(traj_str)
        if not matches:
             # if traj_str != '[]':
                 # print(f"Warning: Regex found no points in trajectory: {traj_str[:50]}...")
             return [], [], None

        for match in matches:
            # Using tuple indexing is slightly faster than named groups
            lon = float(match[0])
            lat = float(match[1])
            ts = int(match[2])

            timestamps.append(ts)
            coords.append((lon, lat))  

        if apply_shift and coords:
            # use list comprehension to convert all points
            # gcj02_to_wgs84 expects (longitude, latitude) and returns (lon_wgs84, lat_wgs84)
            coords = [
                gcj2wgs(lon, lat) for lon, lat in coords # Use the imported gcj2wgs function
            ]

    except (ValueError, IndexError) as e:
        # print(f"Warning: Could not parse point via regex: {traj_str[:50]}... Error: {e}")
        return [], [], None

    line = None
    if CREATE_LINESTRING and len(coords) >= 2:
        try:
            line = LineString(coords)
        except Exception as e:
            # print(f"Warning: Could not create LineString: {e} for coords: {coords[:2]}...")
            line = None
    elif CREATE_LINESTRING and len(coords) == 1:
         pass

    return timestamps, coords, line



In [ ]:
# --- Main Processing ---

df['traj'] = df['traj'].astype(str)
df.loc[df['traj'].str.lower() == 'nan', 'traj'] = '[]'


# Apply the parsing function in PARALLEL
print(f"\nStarting parallel processing using {NUM_WORKERS} workers...")
start_time = time.time()

# Use parallel_apply instead of apply
# Pandarallel automatically chunks the Series and distributes work
results = df['traj'].parallel_apply(parse_trajectory_regex)

# Assign results to new columns
# This part is fast and doesn't need parallelization itself
df[['timestamps', 'coords', 'POLYLINE']] = pd.DataFrame(results.tolist(), index=df.index)

# If LineString creation was disabled, the 'POLYLINE' column will be full of Nones.
if not CREATE_LINESTRING:
    df = df.drop(columns=['POLYLINE'])

end_time = time.time()

# --- Output ---
print(f"\nPreprocessing finished in {end_time - start_time:.2f} seconds.")

print("\nData types of new columns:")
print(df.dtypes)


In [ ]:
len(df)

In [ ]:
# For next step, we want these columns:
# 'TRIP_ID','TAXI_ID','POLYLINE', 'timestamps', 'coords',
df = df.rename(columns={"uid": "TRIP_ID", "did": "TAXI_ID"})

In [ ]:
# length requirement
df.loc[:, 'trajlen'] = df.coords.swifter.apply(lambda traj: len(traj))
df_preprocessed = df[(df.trajlen >= config['min_traj_len']) & (df.trajlen <= config['max_traj_len'])]
print('Preprocessed-rm length. #traj={}'.format(df_preprocessed.shape[0]))

# This one is slow and could use parallelism
from models.utils import lonlat2meters, merc2cell2
df_preprocessed.loc[:, 'merc_seq'] = df_preprocessed.coords.swifter.apply(lambda traj: [list(lonlat2meters(p[0], p[1])) for p in traj])

#df_preprocessed.rename(columns={"coords": "coord_seq"}, inplace=True)

In [ ]:
df_preprocessed

In [ ]:
df_preprocessed_safe = df_preprocessed.copy()
df_preprocessed_safe['POLYLINE'] = df_preprocessed_safe['POLYLINE'].apply(lambda x: wkt.dumps(x))
df_preprocessed_safe.to_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], PEPROCESSED_DATASET_NAME)
) 

In [ ]:
# load df_preprocessed dataset
df_preprocessed = pd.read_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], PEPROCESSED_DATASET_NAME)
)

In [ ]:
from models.utils import lonlat2meters, merc2cell2
df_preprocessed.loc[:, 'merc_seq'] = df_preprocessed.coords.swifter.apply(lambda traj: [list(lonlat2meters(p[0], p[1])) for p in traj])

In [ ]:
# We need timestamps to be a string
df_preprocessed["timestamps"] = df_preprocessed["timestamps"].apply(lambda x: str(x))
create_road_mapping_df(df_preprocessed, config["city"])

#### Do Map Matching now 

we do map matching with FMM through terminal prompts. These are much faster. Afterwards, we get a file mr.txt and continue here.

In [3]:
df_fmm = pd.read_csv(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "mr.txt"), delimiter=";"
)
# Load df_preprocessed again, if not already loaded

In [ ]:
# load df_preprocessed dataset
df_preprocessed = pd.read_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], PEPROCESSED_DATASET_NAME)
)
# df_preprocessed['POLYLINE'] = df_preprocessed['POLYLINE'].apply(lambda x: wkt.loads(x))
df_preprocessed.rename(columns={"coords": "coord_seq"}, inplace=True)


In [4]:
print("Length of df_fmm:", len(df_fmm))
print("Length of df_preprocessed:", len(df_preprocessed))

Length of df_fmm: 1017888
Length of df_preprocessed: 1017888


In [16]:
sorted_ids = sorted(df_fmm.id.tolist())
is_sorted_correctly = sorted_ids == list(range(len(df_fmm)))
print("Are sorted_ids from 0 to len(df_fmm)?", is_sorted_correctly)

Are sorted_ids from 0 to len(df_fmm)? True


In [ ]:
df_preprocessed = df_preprocessed.reset_index(drop=True)
df_merged = merge_preprocessed_and_fmm(df_preprocessed, df_fmm)

1017888 1017888
1017888
Before FMM: 1017888 trips. After FMM: 1017888 trips. Percentage mapped: 1.0


In [24]:
len(df_merged)

1017888

In [25]:
print(f"Before filtering: {len(df)} trips")
df = df[df.cpath.str.len() > 5]
print(f"After filtering (|cpath| > 5): {len(df)} trips")

Before filtering: 1017888 trips
After filtering (|cpath| > 5): 950717 trips


In [35]:
import importlib
import preprocessing.rs_mapping
# Reload the module
importlib.reload(preprocessing.rs_mapping)
# Use the function from the module
df = preprocessing.rs_mapping._calculate_timestamps_for_road_segments(df)

950717it [21:16, 744.81it/s]


In [36]:
df['POLYLINE'] = df['POLYLINE'].apply(lambda x: wkt.dumps(x))
df.to_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], DATASET_NAME)
)

In [ ]:
# load
df = pd.read_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], DATASET_NAME)
)

In [ ]:
# GRID PREPROCESSING - convert lonlat to mercator
#from models.utils import lonlat2meters, merc2cell2
#import swifter
#df['merc_seq'] = df['merc_seq'].apply(lambda x: [list(y) for y in x])

In [37]:
## Split data into train, val, test
train, val, test = generate_train_val_test_split(config['city'], config['seed'])

Train size: 665501, Val size: 190144, Test size: 285216


In [38]:
# Save train, val, test
train.to_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "train", f"train_{config['seed']}.parquet")
)
val.to_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "val", f"val_{config['seed']}.parquet")
)
test.to_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "test", f"test_{config['seed']}.parquet")
)

### Add Road Network Features
- travletime: average traveltime for that road segment
- avg_speed: Average speed driven on that road segmment
- util: average utilization of that road segment (i.e. count of trajectories using that road segment)


In [3]:
##### CALCULATE AVG. TRAVEL TIMES FOR ROAD NETWORK
from collections import defaultdict
import geopandas as gpd

# Load train data
train = pd.read_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "train", f"train_{config['seed']}.parquet")
)
# Load val tdata
val = pd.read_parquet(
    os.path.join(ROOT_DIR, "datasets/trajectory", config["city"], "val", f"val_{config['seed']}.parquet")
)
# join both 
train = pd.concat([train, val], axis=0)

In [4]:
# Load road network 
gdf_edges = gpd.read_file(
        os.path.join(ROOT_DIR, f"datasets/osm/{config['city']}/edges.shp")
    )

In [ ]:

# Create a defaultdict to store total travel time and count for each road ID
road_stats = defaultdict(lambda: {'total_time': 0, 'count': 0})

# Iterate through all rows in the DataFrame
for _, row in tqdm(train.iterrows()):
    cpath = row['cpath']
    timestamps = row['road_timestamps']
    
    # Ensure cpath is a list of integers and timestamps is a list of Unix timestamps
    cpath = eval(cpath) if isinstance(cpath, str) else cpath
    timestamps = eval(timestamps) if isinstance(timestamps, str) else timestamps
    
    # Iterate through road segments
    for i, road_id in enumerate(cpath):
        travel_time = timestamps[i+1] - timestamps[i]
        
        # Update the road_stats dictionary
        road_stats[road_id]['total_time'] += travel_time
        road_stats[road_id]['count'] += 1

# Calculate average travel time for each road ID
average_travel_times = {road_id: stats['total_time'] / stats['count'] 
                        for road_id, stats in road_stats.items()}


# Save to Road Network
gdf_edges['traveltime'] = gdf_edges.index.map(average_travel_times)


855645it [00:45, 18946.85it/s]


In [41]:
# if do not get all road segements, for others calculate traveltime by length/avg_speed
gdf_edges['traveltime'] = gdf_edges.apply(lambda row: row['length'] / (50 * (1000/3600)) if row['traveltime'] == 0 or np.isnan(row['traveltime']) else row['traveltime'], axis=1)
gdf_edges.to_file(os.path.join(ROOT_DIR, f"datasets/osm/{config['city']}/edges.shp"))

In [6]:
from preprocessing.utils import create_edge_emb_mapping, generate_speed_features
speed_df = generate_speed_features(train, gdf_edges)


   id
0   0
1   1
2   2
3   3
4   4


855645it [20:24, 698.68it/s]


In [17]:
df_merged = pd.merge(gdf_edges, speed_df[['util','avg_speed']], left_index=True, right_index=True)

In [28]:
# Next remove edge cases and NaN values

# in edge_df set all avg_speed which are below 10 to 10
# Reasons are: 
# 1. We had 0 values which screwed the training, because we calculated the travel time by dividing the length by the avg_speed, thus got some inf numbers
# 2. Does it make sense that we have avg speed below 10?
df_merged.loc[df_merged['avg_speed'] < 10, 'avg_speed'] = 10
# # Fill NaN values in the avg_speed column based on the highway column
mean_speeds = df_merged.groupby('highway')['avg_speed'].mean() # Calculate the mean avg_speed for each highway
df_merged['avg_speed'] = df_merged.apply(
    lambda row: mean_speeds[row['highway']] if pd.isnull(row['avg_speed']) else row['avg_speed'],
    axis=1
)
fall_back_speed = 25  # or use mean_speeds.mean() if you prefer
df_merged['avg_speed'] = df_merged['avg_speed'].fillna(fall_back_speed)


In [31]:
df_merged.to_file(os.path.join(ROOT_DIR, f"datasets/osm/{config['city']}/edges.shp"), driver='ESRI Shapefile')

### Dynamic Traffic Matrix 

Next we calculate the dynamic traffic matrix needed for the Spatio-Temporal extraction. It contains the avg_speed for each road segment accross the dime of day.

def generate_speed_features(df, edge_df) -> pd.DataFrame:
        """
        Generates features containing average speed, utilization and accelaration
        for each edge i.e road segment.

        Returns:
            pd.DataFrame: features in shape num_edges x features
        """
        rdf = pd.DataFrame({"id": edge_df.fid}, index=edge_df.index)
        # calculate utilization on each edge which is defined as the count an edge is traversed by all trajectories
        seg_seqs = df["cpath"].values
        counter = Counter()
        for seq in seg_seqs:
            counter.update(Counter(seq))

        rdf["util"] = rdf.id.map(counter)


        speed_counter, count_counter = calc_avg_speed(df)
        for j in range(1, 25):
            rdf[f"avg_speed_{j}"] = rdf.id.map(
                {
                    k: (float(speed_counter[j][k]) / count_counter[j][k]) * 111000 * 3.6
                    for k in speed_counter[j]
                }
            )
        return rdf


def calc_avg_speed(data: pd.DataFrame):
    cpaths = data["cpath"].values
    opaths = data["opath"].values
    speeds = data["speed"].values
    times = data["timestamps"].values
    
    # Create a defaultdict of Counters for each hour
    speed_counters = defaultdict(Counter)
    count_counters = defaultdict(Counter)
    
    for opath, cpath, speed, time_stamps in tqdm(zip(opaths, cpaths, speeds, times), total=len(speeds)):
        last_lidx, last_ridx = 0, 0
        for l, r, s, t in zip(opath[0::1], opath[1::1], speed, time_stamps):
            t_hour = datetime.fromtimestamp(t).hour
            
            if s * 111000 * 3.6 >= 200:  # check unrealistic speed values
                continue
            
            lidxs, ridxs = np.where(cpath == l)[0], np.where(cpath == r)[0]
            lidx = lidxs[lidxs >= last_lidx][0]
            ridx = ridxs[(ridxs >= last_ridx) & (ridxs >= lidx)][0]
            
            assert lidx <= ridx
            traversed_edges = cpath[lidx : ridx + 1]
            
            # Update the counters for the specific hour
            speed_counters[t_hour].update(dict(zip(traversed_edges, [s] * len(traversed_edges))))
            count_counters[t_hour].update(dict(zip(traversed_edges, [1] * len(traversed_edges))))
            
            last_lidx, last_ridx = lidx, ridx
    
    return speed_counters, count_counters

In [ ]:

out = generate_speed_features(train, gdf_edges)

In [ ]:
# Interpolate NaN values in the specified columns
avg_speed_columns = [f'avg_speed_{i}' for i in range(1, 25)]
out[avg_speed_columns] = out[avg_speed_columns].interpolate(axis=1, method='linear', limit_direction='both')
out[avg_speed_columns] = out[avg_speed_columns].fillna(method='bfill', axis=1).fillna(method='ffill', axis=1)

In [ ]:
# Fill all remaining NaN values with (we used 40 for porto and sf and 25 for cd)
out[avg_speed_columns] = out[avg_speed_columns].fillna(25)
# Set speed values below 10 to 10
out[avg_speed_columns] = out[avg_speed_columns].clip(lower=10)

In [ ]:
# store as parquet
out.to_parquet(os.path.join(ROOT_DIR,"datasets/transition",city,"traffic_mx.parquet",))